# Predicting Next-Year National CO₂ Emissions (World Bank WDI)

**Research question:** How accurately do economic and energy indicators in year *t* predict national CO₂ emissions per capita in year *t+1*?

Data are downloaded from the original provider (World Bank API). This notebook creates raw CSVs, a processed modeling table, and we will keep working on this notebook to create ML models and visualizations.

In [1]:
from pathlib import Path
import json, time, warnings
from datetime import datetime, timezone
from functools import reduce
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'data').is_dir(), 'Run from project/ or project/notebooks/'
RAW = ROOT / 'data/raw/world_bank_indicators'
PROCESSED = ROOT / 'data/processed'
FIG = ROOT / 'figures'
for d in (RAW, PROCESSED, FIG): d.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

START_YEAR, END_YEAR = 2000, 2020
INDICATORS = {
    'co2_pc': 'EN.GHG.CO2.PC.CE.AR5', # co2 emissions per capita (tons per person)
    'gdp_pc': 'NY.GDP.PCAP.KD', # gdp per capita (constant 2015 US$)
    'fossil_pct': 'EG.USE.COMM.FO.ZS', # fossil fuel consumption 
    'renewable_pct': 'EG.FEC.RNEW.ZS', # renewable energy consumption
    'energy_intensity': 'EG.EGY.PRIM.PP.KD', # energy intensity level of primary energy
    'industry_pct': 'NV.IND.TOTL.ZS', # industry value added, percent of gdp
    'urban_pct': 'SP.URB.TOTL.IN.ZS', # urban population percent of total population
}
PREDICTORS = [x for x in INDICATORS if x != 'co2_pc']

## 1. Retrieve and cache original-provider data

In [2]:
BASE = 'https://api.worldbank.org/v2'
session = requests.Session()
session.headers.update({'User-Agent': 'DS3001-WDI-course-project/1.0'})

# get all pages of world bank api endpoint, returning a list of records
def get_pages(endpoint, params=None):
    params = {**(params or {}), 'format': 'json', 'per_page': 1000}
    rows = []
    page = 1
    while True:
        for attempt in range(4):
            try:
                r = session.get(f'{BASE}/{endpoint}', params={**params, 'page': page}, timeout=60)
                r.raise_for_status()
                payload = r.json()
                if not isinstance(payload, list) or len(payload) != 2 or not isinstance(payload[0], dict):
                    raise ValueError(f'Bad World Bank response: {str(payload)[:200]}')
                break
            except (requests.RequestException, ValueError):
                if attempt == 3: raise
                time.sleep(2 ** attempt)
        meta, batch = payload
        rows.extend(batch or [])
        if page >= int(meta.get('pages', 1)):
            break
        page += 1
    return rows

# get all countries and their metadata, caching to a csv
def load_countries():
    path = RAW / 'countries.csv'
    if path.exists(): return pd.read_csv(path, dtype={'country_code': str})
    records = get_pages('country')
    rows = [{'country_code': r['id'], 'country': r['name'],
             'region': r['region']['value'], 'income_level': r['incomeLevel']['value']}
            for r in records if r['region']['value'] != 'Aggregates']
    frame = pd.DataFrame(rows)
    frame.to_csv(path, index=False)
    return frame

# get all records for a given indicator, store in csv too
def load_indicator(name, code):
    path = RAW / f'{name}.csv'
    if path.exists():
        return pd.read_csv(path, dtype={'country_code': str})
    records = get_pages(f'country/all/indicator/{code}',
                        {'source': 2, 'date': f'{START_YEAR}:{END_YEAR}'})
    frame = pd.DataFrame([{'country_code': r.get('countryiso3code'),
                           'year': int(r['date']), name: r['value']}
                          for r in records])
    if frame.empty: raise ValueError(f'No records returned for {name}: {code}')
    frame.to_csv(path, index=False)
    return frame

countries = load_countries()
tables = {name: load_indicator(name, code) for name, code in INDICATORS.items()}
manifest = {'provider': 'World Bank World Development Indicators',
            'retrieved_utc': datetime.now(timezone.utc).isoformat(),
            'years': [START_YEAR, END_YEAR], 'indicators': INDICATORS,
            'note': 'Date reflects this run; cached files may have been downloaded earlier.'}
(RAW / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Individual countries/economies in metadata:', len(countries))
for name, df in tables.items(): print(name, len(df), 'nonmissing:', df[name].notna().sum())

Individual countries/economies in metadata: 217
co2_pc 5586 nonmissing: 5271
gdp_pc 5586 nonmissing: 5349
fossil_pct 5586 nonmissing: 4090
renewable_pct 5586 nonmissing: 5427
energy_intensity 5586 nonmissing: 5177
industry_pct 5586 nonmissing: 5116
urban_pct 5586 nonmissing: 5565


## 2. Validate keys, clean impossible values and integrate

In [3]:
# one row per country-year. Use ISO3 identifiers, never country names, for merges. 
# preserve rows with incomplete predictor values until preprocessing inside the training pipeline.
valid_codes = set(countries.country_code)
cleaned = []
for name, df in tables.items():
    df = df[['country_code', 'year', name]].copy()
    df = df[df.country_code.isin(valid_codes)]
    df['year'] = pd.to_numeric(df.year, errors='coerce').astype('Int64')
    df[name] = pd.to_numeric(df[name], errors='coerce')
    if df.duplicated(['country_code','year']).any():
        raise ValueError(f'Duplicate country-year keys in {name}')
    if name in ('fossil_pct','renewable_pct','industry_pct','urban_pct'):
        df.loc[~df[name].between(0, 100) & df[name].notna(), name] = np.nan
    if name in ('co2_pc','gdp_pc','energy_intensity'):
        df.loc[df[name] < 0, name] = np.nan
    cleaned.append(df)

panel = reduce(lambda left,right: left.merge(right, on=['country_code','year'],
                                               how='left', validate='one_to_one'), cleaned)
panel = panel.merge(countries, on='country_code', how='inner', validate='many_to_one')
panel = panel.sort_values(['country_code','year']).reset_index(drop=True)
print('All country-years:', len(panel), '| distinct countries:', panel.country_code.nunique())
missingness = panel[list(INDICATORS)].isna().mean().sort_values(ascending=False)
print('Missing values (%):\n', (100 * missingness).round(1))

All country-years: 4557 | distinct countries: 217
Missing values (%):
 fossil_pct          32.1
industry_pct         9.9
energy_intensity     8.5
co2_pc               6.5
gdp_pc               4.7
renewable_pct        3.0
urban_pct            0.0
dtype: float64
